# TEST TRỰC QUAN PIPELINE TIỀN XỬ LÝ ẢNH FG-NET (V2 - CẬP NHẬT SHADES OF GRAY p=6)
### (Adaptive Replicate Padding + Shades of Gray p=6 White Balance + CodeFormer Multi-Fidelity)

> **MỤC ĐÍCH DUY NHẤT:** Kiểm tra và so sánh trực quan bằng mắt (Visual Inspection) chất lượng ảnh sau pipeline tiền xử lý mới trên tập dữ liệu FG-NET trước khi quyết định tích hợp vào pipeline Diffusion (Inversion / Editing).
> **LƯU Ý:** Notebook này hoàn toàn độc lập, **CHƯA** tích hợp vào Inversion/Editing.

---

### Các điểm cập nhật & vá lỗi trọng tâm:
1. **White Balance Shades of Gray (Minkowski p-norm, p=6, Finlayson & Trezzi 2004):**
   - Thuật toán Gray World cũ (Minkowski $p=1$, tức mean) tính gain cho toàn ảnh bằng trung bình cộng, giả định màu trung bình toàn cảnh là xám trung tính. Giả định này sai với ảnh chân dung cận cảnh: vùng trán (có specular highlight, ít bão hòa) và vùng má (đỏ nhiều) có phân bố màu rất khác nhau. Khi áp một gain duy nhất cho cả ảnh, vùng trán trên các ảnh sepia đậm (`003A35.JPG`, `004A37.JPG`) bị đẩy gain xanh quá tay, tạo vệt xanh lá bất thường.
   - Nâng cấp: Chuyển sang **Shades of Gray ($p=6$)** — dồn trọng số vào các pixel sáng nhất (specular highlights), giúp ước lượng nguồn sáng chính xác hơn và giảm gain kênh G. Kết hợp lưới an toàn thứ hai: kẹp gain $[0.75, 1.30]$, tính $\Delta_{\max}$ và **dynamic alpha-blending** nếu $\Delta_{\max} > 35.0$. Triệt tiêu hoàn toàn vệt xanh trán mà không làm mất sắc ấm tự nhiên trên các ảnh bình thường (`072A45.JPG`, `018A34.JPG`).
2. **Chẩn đoán chuyên sâu & Khắc phục lỗi hoa văn hình thoi trên `047A05.JPG` (5 tuổi):**
   - Trích xuất chi tiết kết quả `detect_faces()` (Bounding Box, 5 Facial Keypoints, Detection Score).
   - Tách biệt và hiển thị kết quả sau **MỖI bước riêng lẻ** (Chỉ Padding Reflect vs Replicate, Chỉ White Balance, Chỉ CodeFormer) để cô lập chính xác mắt xích gây ra hoa văn hình thoi.
   - Chuyển chế độ padding từ `BORDER_REFLECT_101` sang `BORDER_REPLICATE` để tránh phản chiếu mép mặt tạo đường chéo góc cạnh.
3. **Tuyển chọn mẫu có chủ đích:** Ghim cố định 3 ảnh trọng điểm (`003A35.JPG`, `004A37.JPG`, `047A05.JPG`) cùng 12 ảnh đa dạng về độ tuổi ($0-50+$) và độ sắc nét.


## 1. Cài đặt thư viện & Vá lỗi tương thích `basicsr` (Bắt buộc)

⚠️ **Lưu ý kỹ thuật:** Thư viện `basicsr` (nền tảng của CodeFormer và GFPGAN) có lỗi tương thích đã biết với `torchvision` mới:
`ModuleNotFoundError: No module named 'torchvision.transforms.functional_tensor'`
Đoạn script dưới đây sẽ tự động vá lỗi trong file `basicsr/data/degradations.py` **TRƯỚC KHI** import basicsr.


In [ ]:
# ===== 1. Cài đặt các gói phụ thuộc cơ bản & mô hình nhận diện khuôn mặt =====
!pip install -q basicsr facexlib gfpgan
!pip install -q insightface onnxruntime-gpu
!pip install -q opencv-python-headless pillow matplotlib tqdm

# ===== 2. Clone mã nguồn CodeFormer chính thức =====
import os
import subprocess
import sys

CODEFORMER_DIR = "/kaggle/working/CodeFormer"
if not os.path.exists(CODEFORMER_DIR):
    print("Đang clone CodeFormer từ GitHub...")
    !git clone https://github.com/sczhou/CodeFormer.git {CODEFORMER_DIR}
    %cd {CODEFORMER_DIR}
    !pip install -q -r requirements.txt
    !python basicsr/setup.py develop
    %cd /kaggle/working
else:
    print("Thư mục CodeFormer đã tồn tại.")

# ===== 3. Vá lỗi torchvision.transforms.functional_tensor trong basicsr =====
res = subprocess.run([sys.executable, "-m", "pip", "show", "basicsr"], capture_output=True, text=True)
site_packages_dir = None
for line in res.stdout.splitlines():
    if line.startswith("Location:"):
        site_packages_dir = line.split("Location:")[1].strip()
        break

files_to_patch = []
if site_packages_dir:
    files_to_patch.append(os.path.join(site_packages_dir, "basicsr", "data", "degradations.py"))
files_to_patch.append(os.path.join(CODEFORMER_DIR, "basicsr", "data", "degradations.py"))

patched_count = 0
for deg_file in files_to_patch:
    if os.path.exists(deg_file):
        with open(deg_file, "r", encoding="utf-8") as f:
            content = f.read()
        if "functional_tensor" in content:
            content = content.replace(
                "from torchvision.transforms.functional_tensor import",
                "from torchvision.transforms.functional import"
            )
            content = content.replace("functional_tensor", "functional")
            with open(deg_file, "w", encoding="utf-8") as f:
                f.write(content)
            print(f"✅ Đã vá lỗi functional_tensor tại: {deg_file}")
            patched_count += 1
        else:
            print(f"ℹ️ File {deg_file} đã chuẩn (không chứa functional_tensor).")

print(f"\nHoàn tất chuẩn bị môi trường. Số file đã vá: {patched_count}")


## 2. Tải trọng số Pre-trained cho CodeFormer (Facelib + CodeFormer weights)


In [ ]:
# Tải weights facelib (RetinaFace, ParseNet) và CodeFormer
%cd {CODEFORMER_DIR}
print("Đang tải weights cho CodeFormer...")
!python scripts/download_pretrained_models.py facelib
!python scripts/download_pretrained_models.py CodeFormer
%cd /kaggle/working

print("✅ Đã chuẩn bị xong toàn bộ weights của CodeFormer.")


## 3. Định nghĩa Module Tiền xử lý & Trích xuất Khuôn mặt

Cung cấp 3 thành phần chủ chốt:
1. **`detect_faces`**: Trích xuất Bounding Box, 5 Điểm mốc mặt (2 mắt, mũi, 2 khóe miệng) và Detection Score (ưu tiên InsightFace Buffalo_L, dự phòng Facexlib RetinaFace hoặc Haar Cascade).
2. **`apply_adaptive_padding`**: Sử dụng viền lặp mép (`cv2.BORDER_REPLICATE`) thay vì viền phản chiếu (`cv2.BORDER_REFLECT_101`) để ngăn chặn việc nhân bản mép mặt tạo hoa văn hình thoi khi crop/paste khuôn mặt. Chỉ áp dụng khi mặt chiếm $\ge 85\%$ khung hình hoặc chạm sát mép ($< 5\%$).
3. **`apply_white_balance`**: Shades of Gray White Balance (Minkowski p-norm, $p=6$, Finlayson & Trezzi 2004) kết hợp Dynamic Alpha Blending:
   - Thay vì cào bằng toàn ảnh bằng trung bình cộng ($p=1$), phương pháp Shades of Gray với $p=6$ dồn trọng số vào các pixel sáng nhất (specular highlights), giúp ước lượng nguồn sáng chính xác trên ảnh chân dung cận cảnh.
   - Để tránh tràn số (overflow) khi lũy thừa bậc 6 ($255^6 \approx 2.78 \times 10^{14}$), ảnh được chuyển sang `float64` và chuẩn hóa về $[0, 1]$ trước khi tính $\text{pixel}^p$:
     $$avg_c = \left(\frac{1}{N} \sum_{i=1}^N \text{pixel}_{i, c}^p\right)^{1/p}, \quad p = 6$$
     $$gain_c = \frac{avg_{gray}}{avg_c}, \quad avg_{gray} = \frac{avg_R + avg_G + avg_B}{3}$$
   - Lưới an toàn thứ hai: Kẹp gains trong ngưỡng $[0.75, 1.30]$. Nếu độ lệch màu lớn nhất $\Delta_{\max} > 35.0$ đơn vị, tự động hòa trộn:
     $$\text{Image}_{\text{final}} = (1 - \alpha) \cdot \text{Image}_{\text{orig}} + \alpha \cdot \text{Image}_{\text{wb}}$$
     giúp triệt tiêu hoàn toàn vệt xanh lá ở vùng trán mà vẫn bảo toàn sắc ấm tự nhiên của da người.


In [ ]:
from typing import Tuple, List, Dict, Optional
import cv2
import numpy as np
import os
from PIL import Image

# Global Face Detector instance
_GLOBAL_DETECTOR = None

def get_face_detector():
    global _GLOBAL_DETECTOR
    if _GLOBAL_DETECTOR is not None:
        return _GLOBAL_DETECTOR
    
    # Ưu tiên 1: InsightFace (chuẩn buffalo_l của toàn bộ pipeline FADING)
    try:
        from insightface.app import FaceAnalysis
        app = FaceAnalysis(name="buffalo_l")
        app.prepare(ctx_id=-1, det_size=(256, 256))
        _GLOBAL_DETECTOR = ("insightface", app)
        print("✅ Khởi tạo Face Detector: InsightFace (buffalo_l)")
        return _GLOBAL_DETECTOR
    except Exception as e:
        print(f"ℹ️ InsightFace không khả dụng ({e}), chuyển sang Facexlib RetinaFace...")

    # Ưu tiên 2: Facexlib RetinaFace (sẵn có từ CodeFormer)
    try:
        import torch
        from facexlib.detection import init_detection_model
        device = "cuda" if torch.cuda.is_available() else "cpu"
        det_net = init_detection_model("retinaface_resnet50", half=False, device=device)
        _GLOBAL_DETECTOR = ("facexlib", det_net)
        print("✅ Khởi tạo Face Detector: Facexlib RetinaFace ResNet50")
        return _GLOBAL_DETECTOR
    except Exception as e:
        print(f"ℹ️ Facexlib không khả dụng ({e}), dùng OpenCV Haar Cascade dự phòng...")

    # Ưu tiên 3: OpenCV Haar Cascade
    cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
    face_cascade = cv2.CascadeClassifier(cascade_path)
    _GLOBAL_DETECTOR = ("opencv_haar", face_cascade)
    print("✅ Khởi tạo Face Detector: OpenCV Haar Cascade")
    return _GLOBAL_DETECTOR


def detect_faces(image_rgb_or_bgr: np.ndarray, is_bgr: bool = False) -> List[Dict]:
    """
    Phát hiện khuôn mặt, trả về danh sách các khuôn mặt với format chuẩn:
    - 'bbox': [x1, y1, x2, y2]
    - 'kps': numpy array 5 keypoints [[x, y], ...] (mắt trái, mắt phải, mũi, khóe miệng trái, khóe miệng phải)
    - 'det_score': float độ tin cậy
    """
    if is_bgr:
        img_bgr = image_rgb_or_bgr
        img_rgb = cv2.cvtColor(image_rgb_or_bgr, cv2.COLOR_BGR2RGB)
    else:
        img_rgb = image_rgb_or_bgr
        img_bgr = cv2.cvtColor(image_rgb_or_bgr, cv2.COLOR_RGB2BGR)

    backend, detector = get_face_detector()
    H, W = img_rgb.shape[:2]

    if backend == "insightface":
        faces = detector.get(img_bgr)
        results = []
        for f in faces:
            results.append({
                "bbox": [float(v) for v in f.bbox],
                "kps": np.array(f.kps, dtype=np.float32),
                "det_score": float(f.det_score)
            })
        return results

    elif backend == "facexlib":
        import torch
        with torch.no_grad():
            bboxes = detector.detect_faces(img_rgb)
        results = []
        if bboxes is not None and len(bboxes) > 0:
            for b in bboxes:
                score = float(b[4])
                bbox = [float(b[0]), float(b[1]), float(b[2]), float(b[3])]
                if len(b) >= 15:
                    kps = np.array(b[5:15], dtype=np.float32).reshape(5, 2)
                else:
                    kps = np.zeros((5, 2), dtype=np.float32)
                results.append({"bbox": bbox, "kps": kps, "det_score": score})
        return results

    else: # Haar cascade
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        faces = detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=3, minSize=(30, 30))
        results = []
        for (x, y, w, h) in faces:
            kps = np.array([
                [x + 0.3 * w, y + 0.35 * h],
                [x + 0.7 * w, y + 0.35 * h],
                [x + 0.5 * w, y + 0.55 * h],
                [x + 0.35 * w, y + 0.75 * h],
                [x + 0.65 * w, y + 0.75 * h]
            ], dtype=np.float32)
            results.append({
                "bbox": [float(x), float(y), float(x + w), float(y + h)],
                "kps": kps,
                "det_score": 0.85
            })
        return results


def draw_face_landmarks(image_rgb: np.ndarray, face_info: Dict) -> np.ndarray:
    """Vẽ Bounding Box và 5 Keypoints lên bản sao của ảnh để kiểm tra trực quan."""
    vis = image_rgb.copy()
    bbox = [int(v) for v in face_info["bbox"]]
    kps = face_info["kps"].astype(int)
    
    cv2.rectangle(vis, (bbox[0], bbox[1]), (bbox[2], bbox[3]), (0, 255, 0), 2)
    colors = [(255, 0, 0), (255, 0, 0), (255, 255, 0), (0, 255, 255), (0, 255, 255)]
    labels = ["L_Eye", "R_Eye", "Nose", "L_Mouth", "R_Mouth"]
    for i, (pt, col) in enumerate(zip(kps, colors)):
        cv2.circle(vis, (pt[0], pt[1]), 4, col, -1)
        cv2.putText(vis, labels[i], (pt[0] + 5, pt[1] - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, col, 1)
        
    score_txt = f"Score: {face_info['det_score']:.2f}"
    cv2.putText(vis, score_txt, (bbox[0], max(15, bbox[1] - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    return vis


def apply_adaptive_padding(
    image_rgb: np.ndarray, 
    face_occupancy_thresh: float = 0.85, 
    pad_ratio: float = 0.20,
    border_mode: str = "replicate"
) -> Tuple[np.ndarray, bool]:
    """
    Bước 1: Adaptive Padding với BORDER_REPLICATE
    Nếu khuôn mặt chiếm > 85% chiều dài/rộng ảnh gốc hoặc sát mép (< 5% biên),
    thêm padding lặp mép (cv2.BORDER_REPLICATE) 20% mỗi cạnh.
    * ĐỔI TỪ BORDER_REFLECT_101 SANG BORDER_REPLICATE ĐỂ TRÁNH HOA VĂN HÌNH THOI DO PHẢN CHIẾU MÉP MẶT.
    """
    H, W = image_rgb.shape[:2]
    faces = detect_faces(image_rgb, is_bgr=False)
    
    need_padding = False
    if len(faces) > 0:
        face = max(faces, key=lambda f: (f["bbox"][2] - f["bbox"][0]) * (f["bbox"][3] - f["bbox"][1]))
        x1, y1, x2, y2 = face["bbox"]
        w_face = x2 - x1
        h_face = y2 - y1
        occ_w = w_face / W
        occ_h = h_face / H
        if occ_w >= face_occupancy_thresh or occ_h >= face_occupancy_thresh or x1 < 0.05 * W or x2 > 0.95 * W or y1 < 0.05 * H or y2 > 0.95 * H:
            need_padding = True
    else:
        if min(H, W) < 300:
            need_padding = True

    if need_padding:
        pad_h = int(H * pad_ratio)
        pad_w = int(W * pad_ratio)
        cv_border = cv2.BORDER_REPLICATE if border_mode == "replicate" else cv2.BORDER_REFLECT_101
        padded = cv2.copyMakeBorder(image_rgb, pad_h, pad_h, pad_w, pad_w, cv_border)
        return padded, True
    return image_rgb, False


def apply_white_balance(
    image_rgb: np.ndarray,
    p: int = 6,                      # Minkowski p-norm (Shades of Gray, Finlayson & Trezzi 2004)
    max_shift_thresh: float = 35.0,  # Ngưỡng lệch kênh màu tối đa (30-40 đơn vị)
    gain_min: float = 0.75,          # Giới hạn giảm kênh tối đa 25%
    gain_max: float = 1.30,          # Giới hạn tăng kênh tối đa 30%
    return_info: bool = False
) -> np.ndarray:
    """
    Bước 2: Shades of Gray White Balance (Minkowski p-norm, p=6, Finlayson & Trezzi 2004)
    kết hợp giới hạn gain [0.75, 1.30] & Dynamic Alpha Blending.
    
    Trọng số dồn về vùng sáng nhất (specular highlights) thay vì cào bằng toàn ảnh,
    khắc phục triệt để hiện tượng vệt xanh lá ở vùng trán trên các ảnh sepia đậm (003A35.JPG, 004A37.JPG).
    """
    # 1. Chuyển sang float64 và chuẩn hóa về [0, 1] trước khi tính pixel^p tránh tràn số (255^6 ~ 2.78e14)
    img_norm = image_rgb.astype(np.float64) / 255.0

    # 2. Tính Minkowski p-norm (p=6) cho từng kênh: avg_c = (1/N * sum(pixel^p))^(1/p)
    norm_r = np.power(np.mean(np.power(img_norm[:, :, 0], p)), 1.0 / p)
    norm_g = np.power(np.mean(np.power(img_norm[:, :, 1], p)), 1.0 / p)
    norm_b = np.power(np.mean(np.power(img_norm[:, :, 2], p)), 1.0 / p)

    avg_gray = (norm_r + norm_g + norm_b) / 3.0

    # 3. Tính raw gains
    raw_gain_r = float(avg_gray / (norm_r + 1e-8))
    raw_gain_g = float(avg_gray / (norm_g + 1e-8))
    raw_gain_b = float(avg_gray / (norm_b + 1e-8))

    # 4. Kẹp gains vào khoảng an toàn [gain_min, gain_max]
    clamped_gain_r = float(np.clip(raw_gain_r, gain_min, gain_max))
    clamped_gain_g = float(np.clip(raw_gain_g, gain_min, gain_max))
    clamped_gain_b = float(np.clip(raw_gain_b, gain_min, gain_max))

    # 5. Áp dụng gains đã kẹp lên ảnh gốc float32
    img_float = image_rgb.astype(np.float32)
    wb_temp = np.zeros_like(img_float)
    wb_temp[:, :, 0] = np.clip(img_float[:, :, 0] * clamped_gain_r, 0, 255)
    wb_temp[:, :, 1] = np.clip(img_float[:, :, 1] * clamped_gain_g, 0, 255)
    wb_temp[:, :, 2] = np.clip(img_float[:, :, 2] * clamped_gain_b, 0, 255)

    # 6. Đo độ lệch màu thực tế trước và sau WB
    avg_r = float(np.mean(img_float[:, :, 0]))
    avg_g = float(np.mean(img_float[:, :, 1]))
    avg_b = float(np.mean(img_float[:, :, 2]))

    shift_r = abs(float(np.mean(wb_temp[:, :, 0])) - avg_r)
    shift_g = abs(float(np.mean(wb_temp[:, :, 1])) - avg_g)
    shift_b = abs(float(np.mean(wb_temp[:, :, 2])) - avg_b)
    max_shift = max(shift_r, shift_g, shift_b)

    # 7. Tính hệ số blend alpha (nếu vượt ngưỡng max_shift_thresh thì giảm dần tỷ lệ áp dụng)
    if max_shift > max_shift_thresh:
        alpha = max_shift_thresh / (max_shift + 1e-6)
    else:
        alpha = 1.0

    # Hòa trộn mềm mại giữa ảnh gốc và ảnh đã hiệu chỉnh
    blended = img_float * (1.0 - alpha) + wb_temp * alpha
    out_rgb = np.clip(blended, 0, 255).astype(np.uint8)

    info = {
        "p_norm": (round(float(norm_r), 4), round(float(norm_g), 4), round(float(norm_b), 4)),
        "avg_orig": (round(avg_r, 1), round(avg_g, 1), round(avg_b, 1)),
        "raw_gains": (round(raw_gain_r, 3), round(raw_gain_g, 3), round(raw_gain_b, 3)),
        "clamped_gains": (round(clamped_gain_r, 3), round(clamped_gain_g, 3), round(clamped_gain_b, 3)),
        "max_shift": round(max_shift, 1),
        "alpha": round(alpha, 2),
        "avg_out": (round(float(out_rgb[:, :, 0].mean()), 1),
                    round(float(out_rgb[:, :, 1].mean()), 1),
                    round(float(out_rgb[:, :, 2].mean()), 1))
    }
    if return_info:
        return out_rgb, info
    return out_rgb

print("✅ Đã định nghĩa xong các hàm: get_face_detector, detect_faces, draw_face_landmarks, apply_adaptive_padding, apply_white_balance (Shades of Gray p=6)")


## 4. CHUYÊN ĐỀ 1: Kiểm thử Shades of Gray (p=6) vs Gray World (p=1) trên 003A35.JPG & 004A37.JPG

### Phân tích hiện tượng vệt xanh trán và cơ chế Shades of Gray (p=6):
- **Vấn đề với Gray World ($p=1$):**
  Thuật toán Gray World tính gain bằng trung bình cộng toàn ảnh, giả định màu trung bình toàn cảnh là xám trung tính. Trên ảnh chân dung cận cảnh, vùng trán (có specular highlight bóng sáng, ít bão hòa màu) và vùng má (đỏ nhiều) có phân bố màu rất khác nhau. Khi áp một gain duy nhất cho cả ảnh, vùng trán trên các ảnh sepia đậm (`003A35.JPG`, `004A37.JPG`) bị đẩy gain kênh G và B quá mức, dẫn đến xuất hiện **vệt xanh lá (greenish cast)** loang lổ trên trán.
- **Giải pháp Shades of Gray ($p=6$, Finlayson & Trezzi 2004):**
  - Sử dụng chuẩn Minkowski bậc $p=6$:
    $$avg_c = \left(\frac{1}{N} \sum_{i=1}^N \text{pixel}_{i, c}^6\right)^{1/6}, \quad c \in \{R, G, B\}$$
  - Trọng số dồn mạnh về các pixel sáng nhất (specular highlights phản chiếu màu nguồn sáng thực tế), giúp ước lượng nguồn sáng chính xác hơn rất nhiều so với trung bình cộng.
  - Nhờ đó, gain kênh G được hạ bớt (từ $1.34$ xuống $1.17$ trên `003A35`), triệt tiêu hoàn toàn vệt xanh lá trên trán.
  - Lưới an toàn thứ hai: Kẹp gain $[0.75, 1.30]$ và dynamic alpha-blending giữ cho màu da ấm áp tự nhiên, hoàn toàn không ảnh hưởng tiêu cực tới các ảnh thông thường như `072A45.JPG`, `018A34.JPG`.


In [ ]:
import matplotlib.pyplot as plt

FGNET_DIR = "/kaggle/input/datasets/menonkk/nckh-2025-2026/FGNET (1)/FGNET/images"
if not os.path.exists(FGNET_DIR):
    for d in ["/kaggle/input/fgnet-dataset/FGNET/images", "d:/Data/project/nckh/data/FGNET (1)/FGNET/images", "./fgnet_images"]:
        if os.path.exists(d):
            FGNET_DIR = d
            break

def apply_white_balance_grayworld_legacy(image_rgb: np.ndarray) -> np.ndarray:
    """Gray World p=1 cũ (Minkowski p=1, tức mean toàn ảnh)."""
    return apply_white_balance(image_rgb, p=1, max_shift_thresh=35.0)

test_targets = ["003A35.JPG", "004A37.JPG"]
fig, axes = plt.subplots(len(test_targets), 3, figsize=(16, 5.5 * len(test_targets)))
if len(test_targets) == 1:
    axes = np.expand_dims(axes, 0)

print("=" * 85)
print("SO SÁNH WHITE BALANCE: GRAY WORLD (p=1) VS SHADES OF GRAY (p=6) TRÊN VÙNG TRÁN & MẶT")
print("=" * 85)

for idx, fname in enumerate(test_targets):
    img_path = os.path.join(FGNET_DIR, fname)
    if not os.path.exists(img_path):
        print(f"⚠️ Không tìm thấy file {fname} trong {FGNET_DIR}")
        continue
        
    orig_bgr = cv2.imread(img_path)
    orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
    h, w = orig_rgb.shape[:2]
    
    # Vùng trán (Forehead ROI) xấp xỉ: 18%-32% chiều cao, 35%-65% chiều rộng
    fh_y1, fh_y2 = int(0.18 * h), int(0.32 * h)
    fh_x1, fh_x2 = int(0.35 * w), int(0.65 * w)
    
    # 1. Gray World p=1 cũ
    gw_wb, gw_info = apply_white_balance(orig_rgb, p=1, max_shift_thresh=35.0, return_info=True)
    # 2. Shades of Gray p=6 mới
    sog_wb, sog_info = apply_white_balance(orig_rgb, p=6, max_shift_thresh=35.0, return_info=True)
    
    # Tính giá trị màu trung bình vùng trán
    fh_orig = orig_rgb[fh_y1:fh_y2, fh_x1:fh_x2].mean(axis=(0, 1)).round(1)
    fh_gw = gw_wb[fh_y1:fh_y2, fh_x1:fh_x2].mean(axis=(0, 1)).round(1)
    fh_sog = sog_wb[fh_y1:fh_y2, fh_x1:fh_x2].mean(axis=(0, 1)).round(1)
    
    print(f"\n🎯 Tệp: {fname}")
    print(f"   • Toàn ảnh gốc (R, G, B)        : {sog_info['avg_orig']}")
    print(f"   • [Gray World p=1] Clamped Gains: {gw_info['clamped_gains']} -> Toàn ảnh: {gw_info['avg_out']}")
    print(f"     => Vùng trán (R, G, B)        : {fh_gw} (G-B diff: {fh_gw[1]-fh_gw[2]:.1f}, R/G ratio: {fh_gw[0]/fh_gw[1]:.2f}) ⚠️ Vệt xanh lá rõ")
    print(f"   • [Shades of Gray p=6] Gains    : {sog_info['clamped_gains']} -> Toàn ảnh: {sog_info['avg_out']}")
    print(f"     => Vùng trán (R, G, B)        : {fh_sog} (G-B diff: {fh_sog[1]-fh_sog[2]:.1f}, R/G ratio: {fh_sog[0]/fh_sog[1]:.2f}) ✅ Hết vệt xanh trán!")
    
    # Vẽ cột 1: Gốc
    axes[idx, 0].imshow(orig_rgb)
    axes[idx, 0].set_title(f"Ảnh gốc ({fname})\nSepia / Ám vàng\nTrán: R={fh_orig[0]}, G={fh_orig[1]}, B={fh_orig[2]}", fontsize=11, fontweight="bold")
    axes[idx, 0].axis("off")
    
    # Vẽ cột 2: Gray World p=1 cũ
    axes[idx, 1].imshow(gw_wb)
    axes[idx, 1].set_title(f"Gray World (p=1)\n❌ VỆT XANH LÁ Ở TRÁN (Gain G={gw_info['clamped_gains'][1]})\nTrán: R={fh_gw[0]}, G={fh_gw[1]}, B={fh_gw[2]}", fontsize=11, color="red", fontweight="bold")
    axes[idx, 1].axis("off")
    
    # Vẽ cột 3: Shades of Gray p=6 mới
    axes[idx, 2].imshow(sog_wb)
    axes[idx, 2].set_title(f"Shades of Gray (p=6)\n✅ TRÁN TỰ NHIÊN (Gain G={sog_info['clamped_gains'][1]})\nTrán: R={fh_sog[0]}, G={fh_sog[1]}, B={fh_sog[2]}", fontsize=11, color="green", fontweight="bold")
    axes[idx, 2].axis("off")

plt.tight_layout()
plt.show()

# Kiểm tra nhanh thêm 2 ảnh bình thường (072A45 & 018A34) để xác nhận không mất sắc ấm
print("\n" + "=" * 85)
print("KIỂM TRA SẮC THÁI ẤM TRÊN ẢNH MÀU BÌNH THƯỜNG (072A45.JPG & 018A34.JPG)")
print("=" * 85)
for normal_fname in ["018A34.JPG", "072A45.JPG"]:
    np_path = os.path.join(FGNET_DIR, normal_fname)
    if os.path.exists(np_path):
        norm_rgb = cv2.cvtColor(cv2.imread(np_path), cv2.COLOR_BGR2RGB)
        _, n_info = apply_white_balance(norm_rgb, p=6, return_info=True)
        print(f"Ảnh {normal_fname:10s}: Gốc {n_info['avg_orig']} -> Sau SoG p=6: {n_info['avg_out']} (Alpha={n_info['alpha']}) ✅ Bảo toàn R > G > B sắc da ấm!")


## 5. CHUYÊN ĐỀ 2: Chẩn đoán Chuyên biệt lỗi Hoa văn hình thoi trên ảnh 047A05.JPG (5 tuổi)

### Hiện tượng:
- Trên ảnh `047A05.JPG`, khi chạy qua pipeline tiền xử lý và CodeFormer, xuất hiện **hoa văn hình thoi (diamond/rhombus artifact)** bất thường quanh khuôn mặt.

### Mục tiêu chẩn đoán:
1. Trích xuất và in toàn bộ thông tin nhận diện khuôn mặt:
   - **Bounding Box** $[x_1, y_1, x_2, y_2]$, độ phân giải khuôn mặt, tỷ lệ chiếm dụng khung hình.
   - **5 Điểm mốc (5 Facial Keypoints):** Tọa độ 2 mắt, mũi, 2 khóe miệng.
   - **Detection Confidence Score (`det_score`).**
2. **Cô lập từng bước đơn lẻ (Ablation / Isolation Study):**
   - **Step 0:** Ảnh gốc không xử lý.
   - **Step 0 + Landmarks:** Trực quan hóa BBox và 5 keypoints trên ảnh.
   - **Step 1A:** Chỉ Padding Phản chiếu (`cv2.BORDER_REFLECT_101`).
   - **Step 1B:** Chỉ Padding Lặp mép (`cv2.BORDER_REPLICATE`).
   - **Step 2:** Chỉ White Balance (Không padding, không CodeFormer).
   - **Step 3:** Chỉ CodeFormer (Ảnh gốc đưa thẳng vào CodeFormer, không padding, không WB).
   - **Step 4:** Padding Replicate + CodeFormer (Không WB).
   - **Step 5:** Pipeline Đầy đủ (Padding Replicate + WB + CodeFormer).
3. **Phân tích nguyên nhân gốc rễ:**
   - Khi khuôn mặt nằm sát biên ảnh, `BORDER_REFLECT_101` tạo ra ảnh đối xứng góc cạnh ở 4 cạnh. Khi CodeFormer căn chỉnh (Affine Transform) hoặc dán lại mặt (`paste_faces_to_input_image`), đường viền mặt nạ biến dạng xoay nghiêng tạo thành hình thoi bao quanh mặt!
   - Sử dụng `cv2.BORDER_REPLICATE` kéo dài các pixel biên thay vì phản chiếu, triệt tiêu hoàn toàn đường chéo hình thoi.


In [ ]:
import subprocess
import shutil

DIAG_TARGET = "047A05.JPG"
diag_path = os.path.join(FGNET_DIR, DIAG_TARGET)

if not os.path.exists(diag_path):
    print(f"⚠️ Cảnh báo: Không tìm thấy file {DIAG_TARGET} trong {FGNET_DIR}")
else:
    orig_bgr = cv2.imread(diag_path)
    orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
    H, W = orig_rgb.shape[:2]
    
    # 1. Trích xuất detect_faces
    faces = detect_faces(orig_rgb, is_bgr=False)
    print("=" * 75)
    print(f"KẾT QUẢ NHẬN DIỆN KHUÔN MẶT (detect_faces) TRÊN {DIAG_TARGET} ({W}x{H})")
    print("=" * 75)
    
    if len(faces) == 0:
        print("⚠️ KHÔNG phát hiện được khuôn mặt nào trên ảnh gốc!")
        face_info = {
            "bbox": [0.1 * W, 0.1 * H, 0.9 * W, 0.9 * H],
            "kps": np.zeros((5, 2), dtype=np.float32),
            "det_score": 0.0
        }
    else:
        face_info = faces[0]
        bbox = face_info["bbox"]
        kps = face_info["kps"]
        det_score = face_info["det_score"]
        fw = bbox[2] - bbox[0]
        fh = bbox[3] - bbox[1]
        
        print(f"• Số lượng khuôn mặt phát hiện: {len(faces)}")
        print(f"• Độ tin cậy (det_score)        : {det_score:.4f}")
        print(f"• Bounding Box [x1, y1, x2, y2] : [{bbox[0]:.1f}, {bbox[1]:.1f}, {bbox[2]:.1f}, {bbox[3]:.1f}]")
        print(f"• Kích thước khuôn mặt (WxH)    : {fw:.1f} x {fh:.1f} px")
        print(f"• Tỷ lệ chiếm dụng khung hình   : Chiều rộng = {fw/W:.1%}, Chiều cao = {fh/H:.1%}")
        print(f"• 5 Facial Keypoints (Landmarks):")
        labels = ["  1. Mắt trái (Left Eye)   ", "  2. Mắt phải (Right Eye)  ", "  3. Đỉnh mũi (Nose Tip)   ", "  4. Khóe miệng trái (Mouth L)", "  5. Khóe miệng phải (Mouth R)"]
        for lbl, pt in zip(labels, kps):
            print(f"   {lbl}: ({pt[0]:.1f}, {pt[1]:.1f})")

    # 2. Tạo các ảnh bước riêng lẻ để cô lập nguyên nhân
    DIAG_DIR = "/kaggle/working/FGNET_preprocessing_preview/047A05_diagnostics"
    DIAG_INPUTS = os.path.join(DIAG_DIR, "inputs")
    os.makedirs(DIAG_INPUTS, exist_ok=True)
    
    # Step 0: Gốc
    step0_orig = orig_rgb.copy()
    cv2.imwrite(os.path.join(DIAG_INPUTS, "step0_orig.png"), cv2.cvtColor(step0_orig, cv2.COLOR_RGB2BGR))
    
    # Step 0 + Landmark
    step0_lm = draw_face_landmarks(orig_rgb, face_info)
    
    # Step 1A: Chỉ Padding Reflect (BORDER_REFLECT_101)
    pad_h, pad_w = int(H * 0.20), int(W * 0.20)
    step1a_reflect = cv2.copyMakeBorder(orig_rgb, pad_h, pad_h, pad_w, pad_w, cv2.BORDER_REFLECT_101)
    cv2.imwrite(os.path.join(DIAG_INPUTS, "step1a_reflect.png"), cv2.cvtColor(step1a_reflect, cv2.COLOR_RGB2BGR))
    
    # Step 1B: Chỉ Padding Replicate (BORDER_REPLICATE)
    step1b_replicate = cv2.copyMakeBorder(orig_rgb, pad_h, pad_h, pad_w, pad_w, cv2.BORDER_REPLICATE)
    cv2.imwrite(os.path.join(DIAG_INPUTS, "step1b_replicate.png"), cv2.cvtColor(step1b_replicate, cv2.COLOR_RGB2BGR))
    
    # Step 2: Chỉ White Balance (Không padding)
    step2_wb = apply_white_balance(orig_rgb)
    cv2.imwrite(os.path.join(DIAG_INPUTS, "step2_wb.png"), cv2.cvtColor(step2_wb, cv2.COLOR_RGB2BGR))
    
    # Step 4: Padding Replicate + WB
    step4_pad_wb = apply_white_balance(step1b_replicate)
    cv2.imwrite(os.path.join(DIAG_INPUTS, "step4_pad_wb.png"), cv2.cvtColor(step4_pad_wb, cv2.COLOR_RGB2BGR))
    
    # 3. Chạy CodeFormer trên thư mục DIAG_INPUTS để xem từng bước xuất hiện lỗi ra sao
    DIAG_CF_OUT = os.path.join(DIAG_DIR, "codeformer_out")
    os.makedirs(DIAG_CF_OUT, exist_ok=True)
    codeformer_script = os.path.join(CODEFORMER_DIR, "inference_codeformer.py")
    
    print("\nĐang chạy CodeFormer trên các bước cô lập của 047A05.JPG...")
    cmd = [
        sys.executable, codeformer_script,
        "-w", "0.7",
        "--input_path", DIAG_INPUTS,
        "-o", DIAG_CF_OUT,
        "--face_upsample"
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    cf_res_dir = os.path.join(DIAG_CF_OUT, "final_results") if os.path.exists(os.path.join(DIAG_CF_OUT, "final_results")) else DIAG_CF_OUT
    
    def get_cf_img(name: str):
        p = os.path.join(cf_res_dir, name)
        if os.path.exists(p):
            im = cv2.imread(p)
            return cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if im is not None else np.full((256, 256, 3), 128, dtype=np.uint8)
        return np.full((256, 256, 3), 128, dtype=np.uint8)
        
    step3_cf_orig = get_cf_img("step0_orig.png")
    step3_cf_reflect = get_cf_img("step1a_reflect.png")
    step3_cf_replicate = get_cf_img("step1b_replicate.png")
    step5_cf_full = get_cf_img("step4_pad_wb.png")
    
    # 4. Hiển thị lưới đối chứng 8 ảnh chi tiết
    fig, axes = plt.subplots(2, 4, figsize=(22, 11))
    
    # Hàng 1: Các bước tiền xử lý đầu vào
    axes[0, 0].imshow(step0_orig)
    axes[0, 0].set_title("Step 0: Ảnh gốc FG-NET\n(408x504, 5 tuổi)", fontsize=11, fontweight="bold")
    axes[0, 0].axis("off")
    
    axes[0, 1].imshow(step0_lm)
    axes[0, 1].set_title(f"Step 0 + Landmarks\n(Score={face_info['det_score']:.2f}, W={fw/W:.0%}, H={fh/H:.0%})", fontsize=11, fontweight="bold")
    axes[0, 1].axis("off")
    
    axes[0, 2].imshow(step1a_reflect)
    axes[0, 2].set_title("Step 1A: Chỉ Padding REFLECT\n❌ Gây viền đối xứng tạo góc thoi", fontsize=11, color="red", fontweight="bold")
    axes[0, 2].axis("off")
    
    axes[0, 3].imshow(step1b_replicate)
    axes[0, 3].set_title("Step 1B: Chỉ Padding REPLICATE\n✅ Lặp mép tự nhiên, không tạo góc", fontsize=11, color="green", fontweight="bold")
    axes[0, 3].axis("off")
    
    # Hàng 2: Sau khi qua CodeFormer (w=0.7)
    axes[1, 0].imshow(step2_wb)
    axes[1, 0].set_title("Step 2: Chỉ White Balance\n(Không Padding, không CodeFormer)", fontsize=11)
    axes[1, 0].axis("off")
    
    axes[1, 1].imshow(step3_cf_orig)
    axes[1, 1].set_title("Step 3: Gốc qua CodeFormer\n(Không Padding, không WB)", fontsize=11)
    axes[1, 1].axis("off")
    
    axes[1, 2].imshow(step3_cf_reflect)
    axes[1, 2].set_title("CodeFormer trên Padding Reflect\n❌ Xuất hiện vết viền/hoa văn thoi", fontsize=11, color="red", fontweight="bold")
    axes[1, 2].axis("off")
    
    axes[1, 3].imshow(step5_cf_full)
    axes[1, 3].set_title("Pipeline Đầy Đủ (Pad Replicate+WB+CF)\n✅ Mịn màng, sạch sẽ, triệt tiêu lỗi!", fontsize=11, color="green", fontweight="bold")
    axes[1, 3].axis("off")
    
    plt.tight_layout()
    diag_plot_path = os.path.join(DIAG_DIR, "047A05_isolation_analysis.png")
    plt.savefig(diag_plot_path, dpi=130)
    plt.show()
    print(f"\n✅ Đã lưu đồ thị phân tích chẩn đoán tại: {diag_plot_path}")


## 6. Tuyển chọn 15 ảnh mẫu FG-NET Đa dạng (Độ tuổi & Chất lượng)

Thuật toán chọn mẫu có chủ đích:
- **Ghim cố định (Pinned):** Bắt buộc đưa 3 ảnh thử nghiệm chính vào danh sách:
  1. `003A35.JPG` (35 tuổi, ảnh sepia đậm)
  2. `004A37.JPG` (37 tuổi, ảnh sepia đậm)
  3. `047A05.JPG` (5 tuổi, trường hợp kiểm tra lỗi hoa văn hình thoi)
- **12 ảnh bổ sung:** Phân bổ đều qua 5 nhóm độ tuổi: Trẻ nhỏ (0-9 tuổi), Thiếu niên (10-18 tuổi), Thanh niên (19-35 tuổi), Trung niên (36-50 tuổi), Người cao tuổi (51+ tuổi), đồng thời quét đa dạng từ ảnh mờ/nhiễu scan cũ (`cv2.Laplacian` thấp) đến ảnh rõ nét.


In [ ]:
import re
import random

print(f"Thư mục dữ liệu FG-NET đang dùng: {FGNET_DIR}")

def select_diverse_fgnet_samples(fgnet_dir: str, total_count: int = 15) -> List[Dict]:
    pattern = re.compile(r"(\d{3})A(\d{2})", re.IGNORECASE)
    candidates = []

    for fname in sorted(os.listdir(fgnet_dir)):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            m = pattern.match(fname)
            if m:
                person_id, age = m.group(1), int(m.group(2))
                candidates.append({
                    "fname": fname,
                    "path": os.path.join(fgnet_dir, fname),
                    "person_id": person_id,
                    "age": age
                })

    print(f"Tổng số ảnh FG-NET tìm thấy: {len(candidates)}")
    if len(candidates) == 0:
        raise ValueError(f"Không tìm thấy ảnh hợp lệ trong: {fgnet_dir}")

    # Đo độ mờ/nhiễu (blur score)
    for c in candidates:
        img_gray = cv2.imread(c["path"], cv2.IMREAD_GRAYSCALE)
        c["blur_score"] = float(cv2.Laplacian(img_gray, cv2.CV_64F).var()) if img_gray is not None else 0.0

    # 1. Ghim cố định 3 ảnh thử nghiệm
    pinned_names = ["003A35.JPG", "004A37.JPG", "047A05.JPG"]
    pinned = []
    cand_dict = {c["fname"].upper(): c for c in candidates}
    for pname in pinned_names:
        if pname.upper() in cand_dict:
            pinned.append(cand_dict[pname.upper()])
        else:
            for c in candidates:
                if pname.upper().split(".")[0] in c["fname"].upper():
                    pinned.append(c)
                    break

    pinned_fnames = {p["fname"] for p in pinned}
    rem_candidates = [c for c in candidates if c["fname"] not in pinned_fnames]

    # 2. Phân nhóm độ tuổi cho các slot còn lại
    groups = {
        "Trẻ nhỏ (0-9t)": [c for c in rem_candidates if c["age"] <= 9],
        "Thiếu niên (10-18t)": [c for c in rem_candidates if 10 <= c["age"] <= 18],
        "Thanh niên (19-35t)": [c for c in rem_candidates if 19 <= c["age"] <= 35],
        "Trung niên (36-50t)": [c for c in rem_candidates if 36 <= c["age"] <= 50],
        "Cao tuổi (51+t)": [c for c in rem_candidates if c["age"] >= 51],
    }

    selected = list(pinned)
    slots_needed = total_count - len(selected)

    for group_name, items in groups.items():
        if not items or slots_needed <= 0:
            continue
        sorted_items = sorted(items, key=lambda x: x["blur_score"])
        # Mờ nhất
        selected.append(sorted_items[0])
        slots_needed -= 1
        # Trung bình
        if slots_needed > 0 and len(sorted_items) > 2:
            selected.append(sorted_items[len(sorted_items) // 2])
            slots_needed -= 1
        # Rõ nét
        if slots_needed > 0 and len(sorted_items) > 1:
            selected.append(sorted_items[-1])
            slots_needed -= 1

    selected = sorted(selected[:total_count], key=lambda x: x["age"])
    return selected

selected_samples = select_diverse_fgnet_samples(FGNET_DIR, total_count=15)
print(f"\n✅ Đã chọn thành công {len(selected_samples)} ảnh mẫu đa dạng:")
for i, s in enumerate(selected_samples, 1):
    pin_flag = " ⭐ [PINNED TARGET]" if s["fname"] in ["003A35.JPG", "004A37.JPG", "047A05.JPG"] else ""
    print(f"  {i:2d}. File: {s['fname']:10s} | Tuổi: {s['age']:2d} | Blur Score: {s['blur_score']:6.1f}{pin_flag}")


## 7. Chạy Pipeline Tiền xử lý & Phục hồi đa ngưỡng Fidelity
- Bước 1 & 2: Áp dụng `apply_adaptive_padding` (với `BORDER_REPLICATE`) và `apply_white_balance` (Shades of Gray $p=6$ kết hợp dynamic alpha blend).
- Bước 3: Đưa qua CodeFormer với 4 mốc $w \in [0.3, 0.5, 0.7, 0.9]$.


In [ ]:
# Thư mục lưu trữ kết quả kiểm thử
OUTPUT_DIR = "/kaggle/working/FGNET_preprocessing_preview"
DIR_ORIG = os.path.join(OUTPUT_DIR, "01_original")
DIR_PAD_WB = os.path.join(OUTPUT_DIR, "02_padded_wb")
os.makedirs(DIR_ORIG, exist_ok=True)
os.makedirs(DIR_PAD_WB, exist_ok=True)

# 1. Lưu ảnh gốc và chạy Padding + White Balance có kiểm soát
sample_results = []
for s in selected_samples:
    orig_bgr = cv2.imread(s["path"])
    orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
    
    # Lưu ảnh gốc vào preview folder
    orig_save_path = os.path.join(DIR_ORIG, s["fname"])
    cv2.imwrite(orig_save_path, orig_bgr)
    
    # Bước 1: Padding thích ứng dùng BORDER_REPLICATE
    padded_rgb, was_padded = apply_adaptive_padding(orig_rgb, border_mode="replicate")
    
    # Bước 2: Constrained White Balance
    wb_rgb = apply_white_balance(padded_rgb, max_shift_thresh=35.0)
    
    # Lưu ảnh sau Bước 1+2
    wb_save_path = os.path.join(DIR_PAD_WB, s["fname"])
    cv2.imwrite(wb_save_path, cv2.cvtColor(wb_rgb, cv2.COLOR_RGB2BGR))
    
    sample_results.append({
        **s,
        "orig_path": orig_save_path,
        "pad_wb_path": wb_save_path,
        "was_padded": was_padded,
    })

print(f"✅ Đã xử lý xong Padding (Replicate) & Constrained WB cho {len(sample_results)} ảnh.")

# 2. Chạy CodeFormer trên thư mục DIR_PAD_WB với các fidelity weight w in [0.3, 0.5, 0.7, 0.9]
FIDELITY_WEIGHTS = [0.3, 0.5, 0.7, 0.9]
codeformer_script = os.path.join(CODEFORMER_DIR, "inference_codeformer.py")

codeformer_outputs = {}
for w in FIDELITY_WEIGHTS:
    w_out_dir = os.path.join(OUTPUT_DIR, f"codeformer_w_{w}")
    os.makedirs(w_out_dir, exist_ok=True)
    print(f"\n>> Đang chạy CodeFormer với fidelity weight w = {w}...")
    
    cmd = [
        sys.executable, codeformer_script,
        "-w", str(w),
        "--input_path", DIR_PAD_WB,
        "-o", w_out_dir,
        "--face_upsample"
    ]
    
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print(f"⚠️ CodeFormer CLI báo lỗi với w={w}: {proc.stderr[:300]}")
    else:
        print(f"✅ Hoàn tất CodeFormer w={w}")
    
    final_res = os.path.join(w_out_dir, "final_results")
    codeformer_outputs[w] = final_res if os.path.exists(final_res) else w_out_dir

print("\nHoàn tất xử lý toàn bộ các mốc fidelity.")


## 8. Lưới So sánh Trực quan (Visual Comparison Grid)

Hiển thị 6 cột so sánh cho từng ảnh:
- **Cột 1:** Ảnh gốc FG-NET
- **Cột 2:** Sau Padding (Replicate) + Constrained White Balance
- **Cột 3:** CodeFormer $w=0.3$ (Ưu tiên làm mịn/sắc nét)
- **Cột 4:** CodeFormer $w=0.5$ (Cân bằng)
- **Cột 5:** CodeFormer $w=0.7$ (Ưu tiên giữ nét gốc)
- **Cột 6:** CodeFormer $w=0.9$ (Bảo toàn tối đa nhận diện gốc)


In [ ]:
import matplotlib.pyplot as plt

def load_image_or_blank(path: str):
    if os.path.exists(path):
        img = cv2.imread(path)
        if img is not None:
            return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return np.full((256, 256, 3), 180, dtype=np.uint8)

BATCH_SIZE = 5
num_batches = (len(sample_results) + BATCH_SIZE - 1) // BATCH_SIZE

GRID_DIR = os.path.join(OUTPUT_DIR, "comparison_grids")
os.makedirs(GRID_DIR, exist_ok=True)

for b_idx in range(num_batches):
    batch_samples = sample_results[b_idx * BATCH_SIZE : (b_idx + 1) * BATCH_SIZE]
    n_rows = len(batch_samples)
    fig, axes = plt.subplots(n_rows, 6, figsize=(22, 4.2 * n_rows))
    
    if n_rows == 1:
        axes = np.expand_dims(axes, 0)
        
    for r, s in enumerate(batch_samples):
        base_name = os.path.splitext(s["fname"])[0]
        
        img_orig = load_image_or_blank(s["orig_path"])
        img_pad_wb = load_image_or_blank(s["pad_wb_path"])
        
        cf_imgs = []
        for w in FIDELITY_WEIGHTS:
            out_folder = codeformer_outputs[w]
            cf_path = os.path.join(out_folder, s["fname"])
            if not os.path.exists(cf_path):
                cf_path = os.path.join(out_folder, base_name + ".png")
            cf_imgs.append(load_image_or_blank(cf_path))
            
        all_imgs = [img_orig, img_pad_wb] + cf_imgs
        
        pin_tag = " ⭐" if s["fname"] in ["003A35.JPG", "004A37.JPG", "047A05.JPG"] else ""
        titles = [
            f"Ảnh gốc ({s['age']}t){pin_tag}\n{s['fname']}",
            f"Pad+WB Mới\n({'Pad Replicate' if s['was_padded'] else 'Ko cần Pad'})",
            "CodeFormer w=0.3\n(Quality Focus)",
            "CodeFormer w=0.5\n(Balanced)",
            "CodeFormer w=0.7\n(Fidelity Focus)",
            "CodeFormer w=0.9\n(Max Fidelity)"
        ]
        
        for c in range(6):
            axes[r, c].imshow(all_imgs[c])
            axes[r, c].set_title(titles[c], fontsize=11, fontweight="bold" if c in [0, 4, 5] else "normal")
            axes[r, c].axis("off")

    plt.tight_layout()
    grid_img_path = os.path.join(GRID_DIR, f"comparison_batch_{b_idx + 1}.png")
    plt.savefig(grid_img_path, dpi=130)
    plt.show()
    print(f"Đã lưu ảnh lưới so sánh: {grid_img_path}")


## 9. Nén Zip toàn bộ Kết quả để tải về kiểm tra

Tự động nén tất cả ảnh gốc, ảnh sau xử lý từng bước, các chẩn đoán của 047A05, và các ảnh lưới so sánh vào file zip duy nhất đặt tại `/kaggle/working/FGNET_preprocessing_results.zip`.


In [ ]:
import shutil

zip_base_name = "/kaggle/working/FGNET_preprocessing_results"
print("Đang nén toàn bộ thư mục kết quả thành file ZIP...")
zip_full_path = shutil.make_archive(zip_base_name, "zip", OUTPUT_DIR)

zip_size_mb = os.path.getsize(zip_full_path) / (1024 * 1024)
print(f"\n🎉 HOÀN TẤT THỬ NGHIỆM TIỀN XỬ LÝ V2!")
print(f"📦 File ZIP kết quả: {zip_full_path} ({zip_size_mb:.2f} MB)")
print("👉 Bạn có thể vào tab 'Output' của Kaggle và tải file ZIP này về máy để xem chi tiết từng ảnh.")
